In [2]:
from scipy.signal import envelope, hilbert
import scipy.io as sio
import pandas as pd
import numpy as np
import sys, os
import torch
project_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from data_classes.decomposition import Extract_Features

In [3]:
phase1_data = sio.loadmat('../data/mine_impact_data_2019.mat')
samples  = pd.DataFrame(phase1_data["x"].T)
labels  = pd.DataFrame(phase1_data["y"].T, columns=["y"])

df = pd.concat([samples, labels], axis=1, join="inner")

df = df.dropna()

In [5]:

shuffled_df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df_X = shuffled_df.iloc[:, :-1]
df_Y = shuffled_df.iloc[:, -1]

data = Extract_Features(
    df_X=df_X,
    df_Y=df_Y,
    feature = "amplitude_envelope",
    frame_size = 128,
    hop_length = 16
)

print(data.get_samples().shape)
print(data.get_labels().shape)


(3309, 2243)
(3309,)


In [6]:
from sklearn.svm import SVC, LinearSVC

svc = SVC(kernel="rbf", C=100)
svc.fit(data.get_samples()[:3000], data.get_labels()[:3000])
print(svc.score(data.get_samples()[3000:], data.get_labels()[3000:]))

0.7411003236245954


In [16]:
import models.classification as classify
import models.loops as loops

train_idx = list(range(0, 3000))
test_idx = list(range(3000,3309))

train_data = torch.utils.data.Subset(data, train_idx)
test_data = torch.utils.data.Subset(data, test_idx)

batch_size = 30

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)

model = classify.MLP(nb_hidden=512, input_dim=data.get_samples().shape[1], output_dim=2, dropout_rate=0.5)

loops.train(model=model, model_path="./model_paths/envelope_1.pth", train_loader=train_loader, batch_size=batch_size, lr=1e-3, weight_decay=1e-2, optim="adam", epochs=20)

loops.test(model_path="./model_paths/envelope_1.pth", test_loader=test_loader)

[INFO] EPOCH: 1/20
Train loss: 0.587909, Train accuracy: 0.6643
[INFO] EPOCH: 2/20
Train loss: 0.550036, Train accuracy: 0.6707
[INFO] EPOCH: 3/20
Train loss: 0.532902, Train accuracy: 0.6857
[INFO] EPOCH: 4/20
Train loss: 0.525074, Train accuracy: 0.7000
[INFO] EPOCH: 5/20
Train loss: 0.515456, Train accuracy: 0.7087
[INFO] EPOCH: 6/20
Train loss: 0.511292, Train accuracy: 0.7060
[INFO] EPOCH: 7/20
Train loss: 0.502460, Train accuracy: 0.7243
[INFO] EPOCH: 8/20
Train loss: 0.496778, Train accuracy: 0.7317
[INFO] EPOCH: 9/20
Train loss: 0.496296, Train accuracy: 0.7307
[INFO] EPOCH: 10/20
Train loss: 0.494519, Train accuracy: 0.7257
[INFO] EPOCH: 11/20
Train loss: 0.481084, Train accuracy: 0.7397
[INFO] EPOCH: 12/20
Train loss: 0.484520, Train accuracy: 0.7367
[INFO] EPOCH: 13/20
Train loss: 0.472113, Train accuracy: 0.7430
[INFO] EPOCH: 14/20
Train loss: 0.463673, Train accuracy: 0.7480
[INFO] EPOCH: 15/20
Train loss: 0.471610, Train accuracy: 0.7487
[INFO] EPOCH: 16/20
Train loss: 0.

In [9]:
## mlp with 1 layer drop or no drop = 74%
## with 2 layer some dropout = 75-76% not much difference with 559 features

## with 1122 features goes upto 77%

In [17]:
import models.classification as classify
import models.loops as loops

train_idx = list(range(0, 3000))
test_idx = list(range(3000,3309))

train_data = torch.utils.data.Subset(data, train_idx)
test_data = torch.utils.data.Subset(data, test_idx)

batch_size = 30

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)

model = classify.MLP_2_layer(nb_hidden=512, input_dim=data.get_samples().shape[1], output_dim=2, dropout_rate=0.5)

loops.train(model=model, model_path="./model_paths/envelope_2.pth", train_loader=train_loader, batch_size=batch_size, lr=1e-3, weight_decay=1e-2, optim="adam", epochs=20)

loops.test(model_path="./model_paths/envelope_2.pth", test_loader=test_loader)

[INFO] EPOCH: 1/20
Train loss: 0.581254, Train accuracy: 0.6677
[INFO] EPOCH: 2/20
Train loss: 0.535894, Train accuracy: 0.6930
[INFO] EPOCH: 3/20
Train loss: 0.513004, Train accuracy: 0.7117
[INFO] EPOCH: 4/20
Train loss: 0.511324, Train accuracy: 0.7137
[INFO] EPOCH: 5/20
Train loss: 0.491471, Train accuracy: 0.7367
[INFO] EPOCH: 6/20
Train loss: 0.477747, Train accuracy: 0.7357
[INFO] EPOCH: 7/20
Train loss: 0.461020, Train accuracy: 0.7580
[INFO] EPOCH: 8/20
Train loss: 0.448080, Train accuracy: 0.7547
[INFO] EPOCH: 9/20
Train loss: 0.422247, Train accuracy: 0.7900
[INFO] EPOCH: 10/20
Train loss: 0.409416, Train accuracy: 0.7943
[INFO] EPOCH: 11/20
Train loss: 0.420337, Train accuracy: 0.7807
[INFO] EPOCH: 12/20
Train loss: 0.400930, Train accuracy: 0.7987
[INFO] EPOCH: 13/20
Train loss: 0.380835, Train accuracy: 0.8203
[INFO] EPOCH: 14/20
Train loss: 0.374453, Train accuracy: 0.8240
[INFO] EPOCH: 15/20
Train loss: 0.358935, Train accuracy: 0.8237
[INFO] EPOCH: 16/20
Train loss: 0.

In [20]:
import models.classification as classify
import models.loops as loops

train_idx = list(range(0, 3000))
test_idx = list(range(3000,3309))

train_data = torch.utils.data.Subset(data, train_idx)
test_data = torch.utils.data.Subset(data, test_idx)

batch_size = 30

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)

model = classify.MLP_3_layer(nb_hidden=1024, input_dim=data.get_samples().shape[1], output_dim=2, dropout_rate=0.6)

loops.train(model=model, model_path="./model_paths/envelope_3.pth", train_loader=train_loader, batch_size=batch_size, lr=1e-3, weight_decay=1e-3, optim="adam", epochs=20)

loops.test(model_path="./model_paths/envelope_3.pth", test_loader=test_loader)

[INFO] EPOCH: 1/20
Train loss: 0.587118, Train accuracy: 0.6647
[INFO] EPOCH: 2/20
Train loss: 0.534114, Train accuracy: 0.6937
[INFO] EPOCH: 3/20
Train loss: 0.514328, Train accuracy: 0.7143
[INFO] EPOCH: 4/20
Train loss: 0.489445, Train accuracy: 0.7353
[INFO] EPOCH: 5/20
Train loss: 0.486324, Train accuracy: 0.7340
[INFO] EPOCH: 6/20
Train loss: 0.457682, Train accuracy: 0.7567
[INFO] EPOCH: 7/20
Train loss: 0.455648, Train accuracy: 0.7587
[INFO] EPOCH: 8/20
Train loss: 0.453010, Train accuracy: 0.7517
[INFO] EPOCH: 9/20
Train loss: 0.431048, Train accuracy: 0.7807
[INFO] EPOCH: 10/20
Train loss: 0.401550, Train accuracy: 0.7977
[INFO] EPOCH: 11/20
Train loss: 0.388571, Train accuracy: 0.8120
[INFO] EPOCH: 12/20
Train loss: 0.372048, Train accuracy: 0.8197
[INFO] EPOCH: 13/20
Train loss: 0.389755, Train accuracy: 0.7967
[INFO] EPOCH: 14/20
Train loss: 0.353186, Train accuracy: 0.8307
[INFO] EPOCH: 15/20
Train loss: 0.350207, Train accuracy: 0.8333
[INFO] EPOCH: 16/20
Train loss: 0.